# 03b - SVM Kernel
Materi Modul 3: mendukung kernel mesin vektor, kernel Gaussian (RBF), alur kerja SVM, dan menerapkan model kernel SVM.

**Kernel trick**: data yang tidak bisa dipisah garis lurus dipetakan ke dimensi lebih tinggi tanpa benar-benar menghitung fitur barunya.

Kernel RBF/Gaussian: $K(x, x') = e^{-\gamma \|x - x'\|^2}$
- `gamma` besar -> pengaruh tiap titik sempit (batas rumit, overfit)
- `gamma` kecil -> pengaruh luas (batas halus)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.kernel_approximation import Nystroem

In [ ]:
def plot_batas(model, X, y, judul=''):
    # menggambar batas keputusan untuk data 2 fitur
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    plt.contourf(xx, yy, zz, alpha=0.3, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=25)
    plt.title(judul)

## 1. Data yang tidak bisa dipisah garis lurus

In [ ]:
X, y = make_moons(n_samples=300, noise=0.25, random_state=42)

fig = plt.figure(figsize=(15, 4))
for i, (kernel, params) in enumerate([('linear', {}), ('poly', {'degree': 3}), ('rbf', {})], start=1):
    plt.subplot(1, 3, i)
    m = SVC(kernel=kernel, **params).fit(X, y)
    plot_batas(m, X, y, f'Kernel {kernel} | akurasi = {m.score(X, y):.3f}')
plt.tight_layout(); plt.show()

## 2. Pengaruh `gamma` dan `C` pada kernel RBF

In [ ]:
fig = plt.figure(figsize=(15, 8))
i = 1
for c in [0.1, 100]:
    for g in [0.1, 1, 50]:
        plt.subplot(2, 3, i)
        m = SVC(kernel='rbf', C=c, gamma=g).fit(X, y)
        plot_batas(m, X, y, f'C={c}, gamma={g}')
        i += 1
plt.tight_layout(); plt.show()

## 3. Alur kerja SVM: scaling -> cari `C` & `gamma` dengan GridSearchCV

In [ ]:
df = pd.read_csv('../datasets/breast_cancer.csv')
Xb, yb = df.drop(columns='target'), df['target']
X_train, X_test, y_train, y_test = train_test_split(Xb, yb, test_size=0.2, stratify=yb, random_state=42)

pipe = make_pipeline(StandardScaler(), SVC())
param = {'svc__C': [0.1, 1, 10, 100], 'svc__gamma': [0.0001, 0.001, 0.01, 0.1]}

grid = GridSearchCV(pipe, param, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

print('Parameter terbaik:', grid.best_params_)
print('Akurasi CV       :', round(grid.best_score_, 4))
print('Akurasi test     :', round(grid.score(X_test, y_test), 4))

In [ ]:
hasil = pd.DataFrame(grid.cv_results_)
tabel = hasil.pivot_table(index='param_svc__C', columns='param_svc__gamma', values='mean_test_score')
plt.imshow(tabel, cmap='viridis')
plt.xticks(range(len(tabel.columns)), tabel.columns); plt.yticks(range(len(tabel.index)), tabel.index)
plt.xlabel('gamma'); plt.ylabel('C'); plt.colorbar(label='Akurasi CV'); plt.title('Grid search SVM RBF')
plt.show()

## 4. Aproksimasi kernel untuk data besar
SVC kernel butuh waktu ~ kuadrat jumlah data. Alternatif: `Nystroem` (aproksimasi kernel RBF) + model linier (`SGDClassifier`).

In [ ]:
Xbig, ybig = make_moons(n_samples=20000, noise=0.25, random_state=0)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xbig, ybig, test_size=0.3, random_state=0)

t = time.time()
svc_rbf = SVC(kernel='rbf', gamma=1).fit(Xb_tr, yb_tr)
waktu_svc = time.time() - t

t = time.time()
approx = make_pipeline(Nystroem(gamma=1, n_components=100, random_state=0), SGDClassifier(loss='hinge', random_state=0)).fit(Xb_tr, yb_tr)
waktu_approx = time.time() - t

print(f'SVC RBF asli         | waktu={waktu_svc:.2f} dtk | akurasi test={svc_rbf.score(Xb_te, yb_te):.4f}')
print(f'Nystroem + SGD       | waktu={waktu_approx:.2f} dtk | akurasi test={approx.score(Xb_te, yb_te):.4f}')

## Catatan
- (apa itu kernel trick, efek gamma dan C, kapan pakai aproksimasi kernel, dll.)